# Module 13: AWS Cloud Services for GenAI

**Companion notebook for**: `13-aws-cloud.html`

## Learning Objectives
- Set up AWS clients using boto3 with proper credential management
- Invoke foundation models (Claude, Llama, Titan) through Amazon Bedrock
- Use Bedrock Knowledge Bases for managed RAG pipelines
- Deploy models with SageMaker JumpStart endpoints
- Build serverless GenAI backends with Lambda + API Gateway + Bedrock
- Use S3 for document storage and DynamoDB for conversation history
- Configure least-privilege IAM policies for GenAI services
- Set up CloudWatch monitoring and cost alerts
- Compare AWS services for different GenAI use cases

## Prerequisites
- AWS account with Bedrock model access enabled
- AWS credentials configured via environment variables or `~/.aws/credentials`
- `boto3` and `sagemaker` Python packages installed

> **Note**: Cells marked with **[REQUIRES AWS ACCESS]** need live AWS credentials to run. All other cells are self-contained demonstrations.

---
## Setup & Dependencies

In [ ]:
%pip install -q boto3 sagemaker

In [ ]:
import os
import json
import time
from pprint import pprint

# ---------- Credential Configuration ----------
# Option 1: Environment variables (recommended for notebooks)
#   export AWS_ACCESS_KEY_ID=AKIA...
#   export AWS_SECRET_ACCESS_KEY=...
#   export AWS_DEFAULT_REGION=us-east-1
#
# Option 2: AWS CLI profile (~/.aws/credentials)
#   aws configure --profile genai-dev
#
# Option 3: IAM Role (automatic on EC2, Lambda, SageMaker)
#   No configuration needed -- boto3 picks up the role automatically.

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

print(f"AWS Region: {AWS_REGION}")
print(f"Credentials source: {'environment variables' if os.environ.get('AWS_ACCESS_KEY_ID') else 'profile / IAM role'}")
print("\nAll imports successful.")

---
## Section 1: Amazon Bedrock -- Client Setup & Model Invocation

Amazon Bedrock is AWS's managed service for accessing foundation models from multiple
providers (Anthropic, Meta, Amazon, Cohere) through a single API. Key advantages:

- **Unified API** -- one interface for Claude, Llama, Titan, Command, and more
- **Enterprise governance** -- IAM policies, VPC endpoints, CloudTrail audit logs
- **Data privacy** -- your data stays in your AWS account and is never used for model training
- **Pay-per-token** -- no upfront commitments for on-demand usage

The tradeoff: Bedrock model versions may lag behind provider direct APIs by days or weeks.

### 1.1 Bedrock Client Setup

In [ ]:
# [REQUIRES AWS ACCESS]
import boto3

# Create the Bedrock runtime client for model invocation
bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)

# List available foundation models (uses the management client)
bedrock_mgmt = boto3.client("bedrock", region_name=AWS_REGION)

response = bedrock_mgmt.list_foundation_models()
models = response["modelSummaries"]

print(f"Total models available: {len(models)}\n")
print("Sample models:")
for m in models[:8]:
    print(f"  {m['modelId']:55s}  {m['providerName']}")

### 1.2 Invoking Claude via Bedrock

Bedrock uses the `invoke_model` API with a JSON body specific to each provider.
For Anthropic models the payload follows the Messages API format with an
additional `anthropic_version` field.

In [ ]:
# [REQUIRES AWS ACCESS]

def invoke_claude_bedrock(prompt: str, max_tokens: int = 1024) -> str:
    """Invoke Claude 3.5 Sonnet via Bedrock and return the text response."""
    response = bedrock.invoke_model(
        modelId="anthropic.claude-3-5-sonnet-20241022-v2:0",
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": max_tokens,
            "messages": [
                {"role": "user", "content": prompt}
            ]
        })
    )

    result = json.loads(response["body"].read())
    return result["content"][0]["text"]


# Test invocation
answer = invoke_claude_bedrock("Explain Amazon Bedrock in 3 sentences.")
print(answer)

### 1.3 Streaming Responses

For chat interfaces, streaming lets you display tokens as they are generated
instead of waiting for the full response.

In [ ]:
# [REQUIRES AWS ACCESS]

def stream_claude_bedrock(prompt: str, max_tokens: int = 1024):
    """Stream a Claude response token-by-token via Bedrock."""
    response = bedrock.invoke_model_with_response_stream(
        modelId="anthropic.claude-3-5-sonnet-20241022-v2:0",
        contentType="application/json",
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": max_tokens,
            "messages": [
                {"role": "user", "content": prompt}
            ]
        })
    )

    for event in response["body"]:
        chunk = json.loads(event["chunk"]["bytes"])
        if chunk["type"] == "content_block_delta":
            print(chunk["delta"]["text"], end="", flush=True)
    print()  # newline at end


stream_claude_bedrock("Write a haiku about cloud computing.")

### 1.4 Bedrock Knowledge Bases -- Managed RAG

Bedrock Knowledge Bases provide a fully managed RAG pipeline:
1. Point at an S3 bucket of documents
2. Bedrock handles chunking, embedding, and vector storage (OpenSearch Serverless)
3. Query with `retrieve_and_generate` -- it retrieves relevant chunks and generates an answer with citations

This eliminates the need to manage a separate vector database.

In [ ]:
# [REQUIRES AWS ACCESS] -- needs a pre-configured Knowledge Base

KNOWLEDGE_BASE_ID = "YOUR_KB_ID_HERE"  # Replace with your KB ID

bedrock_agent = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)

response = bedrock_agent.retrieve_and_generate(
    input={"text": "What is our company's return policy?"},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": KNOWLEDGE_BASE_ID,
            "modelArn": (
                "arn:aws:bedrock:us-east-1::foundation-model/"
                "anthropic.claude-3-5-sonnet-20241022-v2:0"
            )
        }
    }
)

print("Answer:", response["output"]["text"])
print("\nSources:")
for citation in response.get("citations", []):
    for ref in citation["retrievedReferences"]:
        print(f"  {ref['location']['s3Location']['uri']}")

### 1.5 Bedrock Model Comparison

| Model | Provider | Input $/1M tokens | Output $/1M tokens | Best For |
|---|---|---|---|---|
| Claude 3.5 Sonnet | Anthropic | $3.00 | $15.00 | Complex reasoning, coding |
| Claude 3 Haiku | Anthropic | $0.25 | $1.25 | Fast, cost-effective tasks |
| Llama 3 70B | Meta | $2.65 | $3.50 | Open-weight, customizable |
| Titan Text Express | Amazon | $0.20 | $0.60 | Simple tasks, lowest cost |
| Command R+ | Cohere | $3.00 | $15.00 | RAG-optimized |

**Tip**: Use LiteLLM as a unified SDK to call Bedrock with OpenAI-compatible code:
```python
litellm.completion(model="bedrock/anthropic.claude-3-5-sonnet...", messages=[...])
```
This makes your code portable between Bedrock, direct API, and self-hosted models.

---
## Section 2: SageMaker for GenAI

While Bedrock provides access to pre-built models, **SageMaker** gives you full control
over custom model training, fine-tuning, and self-hosted inference.

Key capabilities:
- **JumpStart** -- one-click deployment of popular open models (Llama, Mistral, Falcon)
- **Training Jobs** -- scale to multi-GPU clusters for fine-tuning
- **Endpoints** -- real-time inference with auto-scaling
- **Inference Components** -- host multiple models on the same GPU instance (50-70% cost savings)
- **Model Monitor** -- detect drift and quality degradation

### When to use SageMaker vs. Bedrock
- **Bedrock**: You want a managed API to an existing foundation model
- **SageMaker**: You need to fine-tune, host open-weight models, or have full control over the inference stack

### 2.1 SageMaker JumpStart Deployment

In [ ]:
# [REQUIRES AWS ACCESS] -- deploys a real GPU endpoint (costs ~$1.50/hr)

import sagemaker
from sagemaker.jumpstart.model import JumpStartModel

# Deploy Llama 3 8B via JumpStart
model = JumpStartModel(
    model_id="meta-textgeneration-llama-3-8b-instruct",
    role=sagemaker.get_execution_role(),
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.2xlarge",
    endpoint_name="llama3-jumpstart",
)

# Invoke the deployed model
response = predictor.predict({
    "inputs": "What are the benefits of RAG?",
    "parameters": {"max_new_tokens": 256, "temperature": 0.7}
})
print(response)

In [ ]:
# [REQUIRES AWS ACCESS] -- IMPORTANT: always clean up endpoints to stop billing

# predictor.delete_endpoint()
# print("Endpoint deleted. Billing stopped.")

> **Warning**: SageMaker endpoints bill by the hour even when idle.
> A `ml.g5.2xlarge` costs approximately $1.50/hr (~$1,100/month if left running).
> Always delete endpoints after experimentation. Use lifecycle policies to
> auto-delete idle endpoints in non-production environments.

### 2.2 SageMaker Auto-Scaling Configuration

In [ ]:
# [REQUIRES AWS ACCESS]
# Configure auto-scaling for a SageMaker endpoint (including scale-to-zero)

autoscaling = boto3.client("application-autoscaling", region_name=AWS_REGION)

ENDPOINT_NAME = "llama3-jumpstart"
VARIANT_NAME = "AllTraffic"

resource_id = f"endpoint/{ENDPOINT_NAME}/variant/{VARIANT_NAME}"

# Register the endpoint as a scalable target (min 0 = scale-to-zero)
autoscaling.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=0,   # scale-to-zero for dev/staging
    MaxCapacity=4,
)

# Scale based on invocations per instance
autoscaling.put_scaling_policy(
    PolicyName=f"{ENDPOINT_NAME}-scaling-policy",
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 5.0,  # target 5 invocations per instance
        "PredefinedMetricSpecification": {
            "PredefinedMetricType": "SageMakerVariantInvocationsPerInstance"
        },
        "ScaleInCooldown": 600,
        "ScaleOutCooldown": 60,
    }
)

print(f"Auto-scaling configured: 0-4 instances for {ENDPOINT_NAME}")

---
## Section 3: Serverless GenAI with Lambda

For many GenAI use cases -- chatbot backends with moderate traffic, document processing
pipelines, scheduled report generation -- a **serverless** architecture is the most
cost-effective approach:

- **Lambda** handles application logic (prompt construction, RAG orchestration, response formatting)
- **API Gateway** provides the HTTP endpoint with authentication
- **Bedrock** provides model inference

You pay only when requests are being processed (zero cost during idle periods).

Limitations:
- Lambda timeout: 15 minutes maximum (LLM calls typically 2-15 seconds)
- Streaming requires Lambda Response Streaming or API Gateway WebSocket APIs
- Cold start latency (mitigated with provisioned concurrency)

### 3.1 Lambda Function for Bedrock Chatbot

In [ ]:
# This is the Lambda handler code -- deploy this as a Lambda function.
# It is shown here for reference; it does not run in the notebook context.

LAMBDA_HANDLER_CODE = '''
import json
import boto3

bedrock = boto3.client("bedrock-runtime")

def handler(event, context):
    """Lambda handler for a Bedrock-powered chatbot."""
    body = json.loads(event["body"])
    user_message = body["message"]

    response = bedrock.invoke_model(
        modelId="anthropic.claude-3-haiku-20240307-v1:0",
        contentType="application/json",
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 512,
            "messages": [
                {"role": "user", "content": user_message}
            ]
        })
    )

    result = json.loads(response["body"].read())

    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps({
            "response": result["content"][0]["text"],
            "model": "claude-3-haiku",
            "usage": result["usage"]
        })
    }
'''

print("Lambda handler code (deploy to AWS Lambda):")
print(LAMBDA_HANDLER_CODE)

### 3.2 Deploying the Lambda Function with boto3

In [ ]:
# [REQUIRES AWS ACCESS]
# Programmatic Lambda deployment using boto3

import zipfile
import io

def create_lambda_zip(handler_code: str) -> bytes:
    """Package handler code into a ZIP for Lambda deployment."""
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("lambda_function.py", handler_code)
    return buffer.getvalue()


lambda_client = boto3.client("lambda", region_name=AWS_REGION)

FUNCTION_NAME = "genai-chatbot"
LAMBDA_ROLE_ARN = "arn:aws:iam::123456789012:role/lambda-bedrock-role"  # replace

# Create the function
# lambda_client.create_function(
#     FunctionName=FUNCTION_NAME,
#     Runtime="python3.12",
#     Role=LAMBDA_ROLE_ARN,
#     Handler="lambda_function.handler",
#     Code={"ZipFile": create_lambda_zip(LAMBDA_HANDLER_CODE)},
#     Timeout=30,
#     MemorySize=256,
#     Environment={
#         "Variables": {
#             "BEDROCK_MODEL_ID": "anthropic.claude-3-haiku-20240307-v1:0"
#         }
#     }
# )
# print(f"Lambda function '{FUNCTION_NAME}' created.")

print("Lambda deployment code ready (uncomment to deploy).")
print(f"Function name: {FUNCTION_NAME}")
print(f"Runtime: python3.12")
print(f"Timeout: 30s, Memory: 256MB")

---
## Section 4: Data Services -- S3 & DynamoDB

GenAI applications need several types of data storage:

| Storage Service | Use Case | Key Benefit |
|---|---|---|
| **S3** | Document files (PDFs, images) for RAG pipelines | Unlimited, cheap object storage |
| **OpenSearch Serverless** | Vector embeddings (Bedrock Knowledge Bases) | Fully managed, no capacity planning |
| **Aurora PostgreSQL + pgvector** | Vectors alongside relational data | Single database for structured + vector data |
| **DynamoDB** | Conversation history, session state | Single-digit ms latency, auto-scaling |
| **ElastiCache (Redis)** | Embedding + LLM response caching | Sub-millisecond reads |

### 4.1 S3 for Document Storage (RAG Source)

In [ ]:
# [REQUIRES AWS ACCESS]

s3 = boto3.client("s3", region_name=AWS_REGION)

BUCKET_NAME = "my-genai-documents"  # replace with your bucket


def upload_document(file_path: str, s3_key: str) -> str:
    """Upload a document to S3 for RAG ingestion."""
    s3.upload_file(file_path, BUCKET_NAME, s3_key)
    uri = f"s3://{BUCKET_NAME}/{s3_key}"
    print(f"Uploaded: {uri}")
    return uri


def list_documents(prefix: str = "documents/") -> list:
    """List documents in the RAG source bucket."""
    response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)
    files = []
    for obj in response.get("Contents", []):
        files.append({
            "key": obj["Key"],
            "size_kb": round(obj["Size"] / 1024, 1),
            "modified": obj["LastModified"].isoformat()
        })
    return files


def generate_presigned_url(s3_key: str, expires_in: int = 3600) -> str:
    """Generate a temporary download URL (useful for sharing RAG sources)."""
    url = s3.generate_presigned_url(
        "get_object",
        Params={"Bucket": BUCKET_NAME, "Key": s3_key},
        ExpiresIn=expires_in
    )
    return url


# Demo: list documents in the bucket
# docs = list_documents()
# for doc in docs:
#     print(f"  {doc['key']}  ({doc['size_kb']} KB)")

print("S3 document helpers defined.")
print(f"Target bucket: {BUCKET_NAME}")

### 4.2 DynamoDB for Conversation History

In [ ]:
# [REQUIRES AWS ACCESS]

dynamodb = boto3.resource("dynamodb", region_name=AWS_REGION)
table = dynamodb.Table("conversations")  # must exist in your account


def save_message(session_id: str, role: str, content: str):
    """Save a chat message to DynamoDB with a 7-day TTL."""
    table.put_item(Item={
        "session_id": session_id,
        "timestamp": int(time.time() * 1000),
        "role": role,
        "content": content,
        "ttl": int(time.time()) + 86400 * 7,  # auto-delete after 7 days
    })


def get_history(session_id: str, limit: int = 20) -> list:
    """Retrieve conversation history for a session."""
    response = table.query(
        KeyConditionExpression="session_id = :sid",
        ExpressionAttributeValues={":sid": session_id},
        ScanIndexForward=True,
        Limit=limit,
    )
    return [
        {"role": item["role"], "content": item["content"]}
        for item in response["Items"]
    ]


# Demo usage pattern:
# save_message("session-123", "user", "What is RAG?")
# save_message("session-123", "assistant", "RAG stands for ...")
# history = get_history("session-123")

print("DynamoDB conversation helpers defined.")
print("Table: conversations (partition key: session_id, sort key: timestamp)")

---
## Section 5: Security & IAM

AWS security for GenAI follows the **shared responsibility model**:
- AWS secures the infrastructure
- You secure your application and data

Critical security controls:
1. **IAM policies** -- restrict which models and actions each service/user can access
2. **VPC endpoints** -- keep Bedrock traffic off the public internet
3. **KMS encryption** -- encrypt data at rest and in transit
4. **CloudTrail** -- audit log every model invocation
5. **AWS Config rules** -- enforce compliance automatically

The principle of **least privilege** is especially important: a Lambda function
should only be allowed to invoke the specific Bedrock model IDs it needs.

### 5.1 IAM Policy: Least-Privilege Bedrock Access

In [ ]:
# IAM policy document -- apply this to Lambda execution roles or IAM users

bedrock_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowSpecificBedrockModels",
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream"
            ],
            "Resource": [
                "arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-3-haiku*",
                "arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-3-5-sonnet*"
            ]
        },
        {
            "Sid": "DenyExpensiveModelsExceptMLTeam",
            "Effect": "Deny",
            "Action": "bedrock:InvokeModel",
            "Resource": "arn:aws:bedrock:*::foundation-model/anthropic.claude-3-opus*",
            "Condition": {
                "StringNotEquals": {
                    "aws:PrincipalTag/team": "ml-research"
                }
            }
        }
    ]
}

print("Bedrock IAM Policy (least-privilege):")
print(json.dumps(bedrock_policy, indent=2))

### 5.2 Lambda Execution Role for GenAI

In [ ]:
# Complete IAM role configuration for a GenAI Lambda function

lambda_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }
    ]
}

lambda_permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BedrockInvoke",
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream"
            ],
            "Resource": [
                "arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-3-haiku*"
            ]
        },
        {
            "Sid": "DynamoDBConversations",
            "Effect": "Allow",
            "Action": [
                "dynamodb:PutItem",
                "dynamodb:Query"
            ],
            "Resource": "arn:aws:dynamodb:us-east-1:123456789012:table/conversations"
        },
        {
            "Sid": "S3ReadDocuments",
            "Effect": "Allow",
            "Action": ["s3:GetObject"],
            "Resource": "arn:aws:s3:::my-genai-documents/*"
        },
        {
            "Sid": "CloudWatchLogs",
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents"
            ],
            "Resource": "arn:aws:logs:us-east-1:123456789012:*"
        }
    ]
}

print("Lambda Trust Policy (who can assume this role):")
print(json.dumps(lambda_trust_policy, indent=2))
print("\nLambda Permissions Policy (what the role can do):")
print(json.dumps(lambda_permissions_policy, indent=2))

### 5.3 VPC Endpoint for Private Bedrock Access

In [ ]:
# [REQUIRES AWS ACCESS]
# Create a VPC endpoint so Bedrock traffic never leaves your VPC

ec2 = boto3.client("ec2", region_name=AWS_REGION)

VPC_ID = "vpc-0123456789abcdef0"           # replace
SUBNET_IDS = ["subnet-aaa", "subnet-bbb"]  # replace
SECURITY_GROUP_ID = "sg-0123456789abcdef0"  # replace

# Uncomment to create:
# response = ec2.create_vpc_endpoint(
#     VpcEndpointType="Interface",
#     ServiceName=f"com.amazonaws.{AWS_REGION}.bedrock-runtime",
#     VpcId=VPC_ID,
#     SubnetIds=SUBNET_IDS,
#     SecurityGroupIds=[SECURITY_GROUP_ID],
#     PrivateDnsEnabled=True,
# )
# print(f"VPC Endpoint created: {response['VpcEndpoint']['VpcEndpointId']}")

print("VPC endpoint configuration ready.")
print("Service: com.amazonaws.us-east-1.bedrock-runtime")
print("Effect: All Bedrock API calls stay within your VPC (no public internet).")

---
## Section 6: Monitoring with CloudWatch

Monitoring GenAI workloads requires tracking:
- **Latency** -- model response times (P50, P95, P99)
- **Error rates** -- throttling, model errors, timeouts
- **Token usage** -- input/output tokens per request
- **Costs** -- daily/weekly spend alerts
- **Quality** -- response quality degradation (via custom metrics)

### 6.1 CloudWatch Cost Alarm for Bedrock

In [ ]:
# [REQUIRES AWS ACCESS]

cloudwatch = boto3.client("cloudwatch", region_name=AWS_REGION)

# Set up a cost alarm that triggers at $100/day
cloudwatch.put_metric_alarm(
    AlarmName="BedrockDailyCostAlarm",
    MetricName="EstimatedCharges",
    Namespace="AWS/Billing",
    Statistic="Maximum",
    Period=86400,           # 24 hours
    EvaluationPeriods=1,
    Threshold=100.0,        # alert at $100/day
    ComparisonOperator="GreaterThanThreshold",
    Dimensions=[
        {"Name": "ServiceName", "Value": "Amazon Bedrock"}
    ],
    AlarmActions=[
        "arn:aws:sns:us-east-1:123456789012:cost-alerts"  # replace with your SNS topic
    ],
)

print("CloudWatch alarm 'BedrockDailyCostAlarm' created.")
print("Triggers when Bedrock charges exceed $100/day.")

### 6.2 Custom CloudWatch Metrics for GenAI

In [ ]:
# Publish custom metrics for LLM observability
# Embed this in your application code alongside Bedrock calls

def publish_genai_metrics(
    cloudwatch_client,
    model_id: str,
    latency_ms: float,
    input_tokens: int,
    output_tokens: int,
    success: bool = True
):
    """Publish custom GenAI metrics to CloudWatch."""
    cloudwatch_client.put_metric_data(
        Namespace="GenAI/Application",
        MetricData=[
            {
                "MetricName": "InvocationLatency",
                "Value": latency_ms,
                "Unit": "Milliseconds",
                "Dimensions": [{"Name": "ModelId", "Value": model_id}]
            },
            {
                "MetricName": "InputTokens",
                "Value": input_tokens,
                "Unit": "Count",
                "Dimensions": [{"Name": "ModelId", "Value": model_id}]
            },
            {
                "MetricName": "OutputTokens",
                "Value": output_tokens,
                "Unit": "Count",
                "Dimensions": [{"Name": "ModelId", "Value": model_id}]
            },
            {
                "MetricName": "InvocationErrors",
                "Value": 0.0 if success else 1.0,
                "Unit": "Count",
                "Dimensions": [{"Name": "ModelId", "Value": model_id}]
            }
        ]
    )


# Usage example (not executed -- requires AWS access):
# publish_genai_metrics(
#     cloudwatch_client=cloudwatch,
#     model_id="anthropic.claude-3-haiku",
#     latency_ms=1250.0,
#     input_tokens=350,
#     output_tokens=180,
#     success=True
# )

print("Custom CloudWatch metrics publisher defined.")
print("Metrics: InvocationLatency, InputTokens, OutputTokens, InvocationErrors")
print("Namespace: GenAI/Application")

---
## Section 7: Cloud Architecture Patterns

Below are text-based architecture diagrams for common GenAI patterns on AWS.

### 7.1 Serverless RAG Chatbot Architecture

In [ ]:
architecture_serverless_rag = """
=================================================================
          SERVERLESS RAG CHATBOT -- AWS ARCHITECTURE
=================================================================

  User (Browser / Mobile)
    |
    | HTTPS
    v
  +---------------------------+
  |  Amazon API Gateway       |   <-- REST or WebSocket API
  |  (auth: Cognito / API key)|       Rate limiting, throttling
  +---------------------------+
    |
    v
  +---------------------------+
  |  AWS Lambda               |   <-- Application logic
  |  - Prompt construction    |       Timeout: 30s
  |  - RAG orchestration      |       Memory: 256MB
  |  - Response formatting    |
  +---------------------------+
    |           |           |
    v           v           v
  +-------+  +-------+  +------------------+
  |Bedrock|  |DynamoDB|  |Bedrock Knowledge |
  |Runtime|  |        |  |Base              |
  | Claude|  |Chat    |  | S3 docs          |
  | Haiku |  |History |  | OpenSearch vector|
  +-------+  +-------+  +------------------+
                              |
                              v
                         +---------+
                         |   S3    |
                         |Documents|
                         |(PDFs,   |
                         | TXT,    |
                         | HTML)   |
                         +---------+

  Cost: ~$0 when idle, scales to thousands of concurrent users.
  Typical per-request cost: $0.001 - $0.01 (model tokens).
=================================================================
"""

print(architecture_serverless_rag)

### 7.2 Production GenAI Platform Architecture

In [ ]:
architecture_production = """
======================================================================
       PRODUCTION GenAI PLATFORM -- AWS ARCHITECTURE
======================================================================

              +-------------------+
              |    CloudFront     |   <-- CDN, WAF, DDoS protection
              +-------------------+
                      |
              +-------------------+
              |   ALB / API GW    |   <-- Load balancing, auth
              +-------------------+
                /          \\
               v            v
    +--------------+  +--------------+
    | ECS Fargate  |  | Lambda       |   <-- Compute layer
    | (long-running|  | (event-      |       Choose based on
    |  agents)     |  |  driven)     |       workload pattern
    +--------------+  +--------------+
          |      |           |
          v      v           v
    +--------+ +--------+ +--------+
    |Bedrock | |SageMaker| |Bedrock |
    |Runtime | |Endpoint | |KB (RAG)|
    |(Claude)| |(Llama)  | |        |
    +--------+ +--------+ +--------+
                   |           |
    Data Layer:     v           v
    +---------+ +---------+ +-----------+
    |DynamoDB | |   S3    | |OpenSearch  |
    |Sessions | |Documents| |Serverless  |
    |& State  | |& Models | |(Vectors)   |
    +---------+ +---------+ +-----------+

    Observability:                  Security:
    +------------+                  +------------+
    |CloudWatch  |                  |IAM (least  |
    |X-Ray       |                  | privilege)  |
    |Bedrock     |                  |VPC Endpoints|
    | Model Logs |                  |KMS, CloudTrail|
    +------------+                  +------------+
======================================================================
"""

print(architecture_production)

### 7.3 Document Processing Pipeline Architecture

In [ ]:
architecture_doc_pipeline = """
======================================================================
     DOCUMENT PROCESSING PIPELINE -- EVENT-DRIVEN ARCHITECTURE
======================================================================

  Upload (API / Console)
    |
    v
  +-----------+    S3 Event     +----------------+
  |  S3 Bucket| -------------> | EventBridge /   |
  |  (raw     |  (ObjectCreated)|  S3 Notification|
  |   docs)   |                +----------------+
  +-----------+                        |
                                       v
                              +------------------+
                              | Step Functions   |  <-- Orchestration
                              | State Machine    |
                              +------------------+
                              /        |         \\
                             v         v          v
                      +--------+ +--------+ +---------+
                      |Lambda 1| |Lambda 2| |Lambda 3 |
                      |Extract | |Chunk & | |Generate |
                      |Text    | |Embed   | |Summary  |
                      |(Textract)|(Bedrock)| |(Bedrock)|
                      +--------+ +--------+ +---------+
                           |          |           |
                           v          v           v
                      +--------+ +--------+ +---------+
                      |S3      | |OpenSearch| |DynamoDB |
                      |(parsed)| |(vectors) | |(metadata|
                      +--------+ +--------+ | summary) |
                                             +---------+

  Cost model: Pay per document processed. Zero cost when idle.
  Throughput: Parallel processing via Step Functions Map state.
======================================================================
"""

print(architecture_doc_pipeline)

---
## Section 8: Cost Optimization Strategies

GenAI costs on AWS can escalate quickly without proper controls.

### Main Cost Drivers
1. **Model inference** -- per-token (Bedrock) or per-hour (SageMaker endpoints)
2. **GPU instances** -- g5.2xlarge at ~$1.50/hr = ~$1,100/month if always on
3. **Vector database** -- OpenSearch Serverless has minimum charges
4. **Data transfer** -- cross-region and internet egress

### Cost Optimization Summary

| Strategy | Savings | Effort |
|---|---|---|
| Use smallest effective model (Haiku before Sonnet) | 80-90% | Low -- just test quality |
| Cache identical prompts (Redis/ElastiCache) | 30-70% | Medium |
| Provisioned Throughput for high volume | 30-50% | Low -- commitment required |
| Auto-scale SageMaker to zero off-hours | 50-70% | Medium |
| Prompt engineering (shorter prompts) | 20-40% | Medium |

In [ ]:
# Cost estimator for Bedrock usage

BEDROCK_PRICING = {
    "claude-3.5-sonnet": {"input": 3.00, "output": 15.00},
    "claude-3-haiku":    {"input": 0.25, "output": 1.25},
    "llama-3-70b":       {"input": 2.65, "output": 3.50},
    "titan-text-express":{"input": 0.20, "output": 0.60},
    "command-r-plus":    {"input": 3.00, "output": 15.00},
}


def estimate_monthly_cost(
    model: str,
    requests_per_day: int,
    avg_input_tokens: int = 500,
    avg_output_tokens: int = 300
) -> dict:
    """Estimate monthly Bedrock cost for a given usage pattern."""
    pricing = BEDROCK_PRICING.get(model)
    if not pricing:
        return {"error": f"Unknown model: {model}"}

    daily_input_tokens = requests_per_day * avg_input_tokens
    daily_output_tokens = requests_per_day * avg_output_tokens

    daily_input_cost = (daily_input_tokens / 1_000_000) * pricing["input"]
    daily_output_cost = (daily_output_tokens / 1_000_000) * pricing["output"]
    daily_total = daily_input_cost + daily_output_cost
    monthly_total = daily_total * 30

    return {
        "model": model,
        "requests_per_day": requests_per_day,
        "daily_cost": round(daily_total, 2),
        "monthly_cost": round(monthly_total, 2),
        "cost_per_request": round(daily_total / requests_per_day, 5),
    }


# Compare costs across models for 1,000 requests/day
print("Monthly cost comparison (1,000 requests/day, 500 in / 300 out tokens):")
print(f"{'Model':<22s} {'Daily':>8s} {'Monthly':>10s} {'Per Request':>12s}")
print("-" * 55)

for model_name in BEDROCK_PRICING:
    est = estimate_monthly_cost(model_name, requests_per_day=1000)
    print(f"{est['model']:<22s} ${est['daily_cost']:>6.2f} ${est['monthly_cost']:>8.2f} ${est['cost_per_request']:>10.5f}")

In [ ]:
# Model routing: use the cheapest model that meets quality requirements

def route_to_model(prompt: str, complexity: str = "auto") -> str:
    """
    Route a request to the most cost-effective model based on complexity.

    Routing strategy:
    - simple: Short answers, classification, extraction -> Haiku ($0.25/1M input)
    - moderate: Summarization, Q&A, moderate reasoning -> Haiku or Sonnet
    - complex: Multi-step reasoning, code generation, analysis -> Sonnet ($3.00/1M input)
    """
    if complexity == "auto":
        # Simple heuristic: route based on prompt length and keywords
        word_count = len(prompt.split())
        complex_keywords = ["analyze", "compare", "debug", "refactor", "explain why",
                           "step by step", "write code", "design"]
        has_complex_keyword = any(kw in prompt.lower() for kw in complex_keywords)

        if word_count < 50 and not has_complex_keyword:
            complexity = "simple"
        elif has_complex_keyword or word_count > 200:
            complexity = "complex"
        else:
            complexity = "moderate"

    model_map = {
        "simple": "anthropic.claude-3-haiku-20240307-v1:0",
        "moderate": "anthropic.claude-3-haiku-20240307-v1:0",
        "complex": "anthropic.claude-3-5-sonnet-20241022-v2:0",
    }

    selected = model_map.get(complexity, model_map["moderate"])
    return selected


# Demo routing decisions
test_prompts = [
    "What is the capital of France?",
    "Summarize this document in 3 bullet points.",
    "Analyze this code for security vulnerabilities and explain why each is dangerous. Suggest fixes step by step.",
]

print("Model routing demo:")
for prompt in test_prompts:
    model = route_to_model(prompt)
    short_model = model.split("/")[-1].split("-20")[0] if "-20" in model else model
    print(f"  Prompt: \"{prompt[:60]}...\"")
    print(f"  -> {short_model}")
    print()

### 8.1 AWS Budget Alert Setup

In [ ]:
# [REQUIRES AWS ACCESS]
# Set up AWS Budgets alerts BEFORE deploying any GenAI workload

budgets = boto3.client("budgets", region_name=AWS_REGION)

ACCOUNT_ID = "123456789012"  # replace
ALERT_EMAIL = "team@example.com"  # replace

# Uncomment to create:
# budgets.create_budget(
#     AccountId=ACCOUNT_ID,
#     Budget={
#         "BudgetName": "GenAI-Monthly-Budget",
#         "BudgetLimit": {"Amount": "500", "Unit": "USD"},
#         "TimeUnit": "MONTHLY",
#         "BudgetType": "COST",
#         "CostFilters": {
#             "Service": ["Amazon Bedrock", "Amazon SageMaker"]
#         }
#     },
#     NotificationsWithSubscribers=[
#         {
#             "Notification": {
#                 "NotificationType": "ACTUAL",
#                 "ComparisonOperator": "GREATER_THAN",
#                 "Threshold": 50.0,  # alert at 50% of budget
#             },
#             "Subscribers": [{"SubscriptionType": "EMAIL", "Address": ALERT_EMAIL}]
#         },
#         {
#             "Notification": {
#                 "NotificationType": "ACTUAL",
#                 "ComparisonOperator": "GREATER_THAN",
#                 "Threshold": 80.0,  # alert at 80% of budget
#             },
#             "Subscribers": [{"SubscriptionType": "EMAIL", "Address": ALERT_EMAIL}]
#         },
#         {
#             "Notification": {
#                 "NotificationType": "ACTUAL",
#                 "ComparisonOperator": "GREATER_THAN",
#                 "Threshold": 100.0,  # alert at 100% of budget
#             },
#             "Subscribers": [{"SubscriptionType": "EMAIL", "Address": ALERT_EMAIL}]
#         },
#     ]
# )

print("Budget alert configuration ready.")
print("Budget: $500/month for Bedrock + SageMaker")
print("Alerts at: 50%, 80%, 100% of budget")
print("\nBest practice: Also add a hard stop (Lambda + EventBridge) that")
print("disables resources if spending exceeds 150% of budget.")

---
## Section 9: Comparing AWS Services for GenAI Use Cases

Choosing the right AWS service depends on your use case, team expertise, and requirements.

In [ ]:
# Decision matrix: which AWS service for which GenAI use case

comparison = [
    {
        "use_case": "Chatbot with pre-built models",
        "recommended": "Bedrock + Lambda",
        "why": "Managed models, serverless, pay-per-token, zero infrastructure",
        "cost_profile": "Low (usage-based)"
    },
    {
        "use_case": "RAG over company documents",
        "recommended": "Bedrock Knowledge Bases",
        "why": "Fully managed RAG: chunking, embedding, retrieval -- no vector DB to manage",
        "cost_profile": "Medium (storage + queries)"
    },
    {
        "use_case": "Fine-tuned model for specific domain",
        "recommended": "SageMaker Training + Endpoints",
        "why": "Full control over training, custom model artifacts, GPU selection",
        "cost_profile": "High (GPU hours for training + hosting)"
    },
    {
        "use_case": "Open-weight model hosting (Llama, Mistral)",
        "recommended": "SageMaker JumpStart or ECS + vLLM",
        "why": "One-click deploy with JumpStart; full control with ECS/vLLM",
        "cost_profile": "High (dedicated GPU instances)"
    },
    {
        "use_case": "Document processing pipeline",
        "recommended": "Step Functions + Lambda + Bedrock",
        "why": "Event-driven, parallel processing, zero idle cost",
        "cost_profile": "Low (per-document cost)"
    },
    {
        "use_case": "Multi-model routing (cheap + expensive)",
        "recommended": "Bedrock (multiple models) + custom router",
        "why": "Route simple queries to Haiku, complex to Sonnet -- same API",
        "cost_profile": "Optimized (80%+ savings vs. always using Sonnet)"
    },
    {
        "use_case": "Real-time agent with tool use",
        "recommended": "Bedrock Agents or ECS Fargate + LangGraph",
        "why": "Bedrock Agents for managed; ECS for custom agent frameworks",
        "cost_profile": "Medium-High (multiple LLM calls per agent run)"
    },
    {
        "use_case": "Image / multimodal generation",
        "recommended": "Bedrock (Stability AI, Titan Image)",
        "why": "Managed image generation, no GPU management",
        "cost_profile": "Medium (per-image pricing)"
    },
]

print(f"{'Use Case':<40s} {'Recommended Service':<35s} {'Cost'}")
print("=" * 100)
for row in comparison:
    print(f"{row['use_case']:<40s} {row['recommended']:<35s} {row['cost_profile']}")
    print(f"  Why: {row['why']}")
    print()

In [ ]:
# Decision tree: Bedrock vs. SageMaker vs. ECS self-hosted

decision_tree = """
================== WHICH AWS SERVICE SHOULD I USE? ==================

Do you need a CUSTOM or FINE-TUNED model?
  |
  +-- NO: Do you want a MANAGED API (no infrastructure)?
  |    |
  |    +-- YES --> Amazon BEDROCK
  |    |           - Unified API for Claude, Llama, Titan
  |    |           - Pay per token, no servers
  |    |           - Built-in Knowledge Bases, Agents, Guardrails
  |    |
  |    +-- NO: Do you need full control over the inference stack?
  |         |
  |         +-- YES --> ECS Fargate + vLLM/TGI
  |                     - Custom containers, full control
  |                     - Scale with ECS auto-scaling
  |
  +-- YES: Do you need managed TRAINING infrastructure?
       |
       +-- YES --> SageMaker Training + Endpoints
       |           - Managed Jupyter, training jobs, endpoints
       |           - JumpStart for quick open-model deployment
       |           - Model Monitor for drift detection
       |
       +-- NO --> Bedrock Custom Model / Fine-Tuning
                  - Limited model selection for fine-tuning
                  - Simpler but less flexible than SageMaker

====================================================================
"""

print(decision_tree)

---
## Summary

In this notebook we covered:

1. **Amazon Bedrock** -- Client setup, model invocation (sync + streaming), Knowledge Bases for managed RAG, and model pricing comparison
2. **SageMaker** -- JumpStart deployment of open-weight models, auto-scaling configuration, and cost management for GPU endpoints
3. **Serverless GenAI** -- Lambda handler for Bedrock chatbot, deployment patterns, and API Gateway integration
4. **Data Services** -- S3 for document storage, DynamoDB for conversation history with TTL
5. **Security & IAM** -- Least-privilege Bedrock policies, Lambda execution roles, VPC endpoints for private access
6. **Monitoring** -- CloudWatch cost alarms, custom GenAI metrics (latency, tokens, errors)
7. **Architecture Patterns** -- Serverless RAG chatbot, production platform, and document processing pipeline diagrams
8. **Cost Optimization** -- Model cost comparison, intelligent routing, budget alerts
9. **Service Comparison** -- Decision matrix for choosing Bedrock vs. SageMaker vs. ECS for different use cases

**Key takeaway**: Start with Bedrock for managed model access, add SageMaker only when you need custom training or full inference control, and always set up budget alerts before deploying any GenAI workload.

**Next module**: `14-n8n-no-code.html` -- Building GenAI workflows with n8n (no-code automation).